In [ ]:
from sklearn.metrics import make_scorer, f1_score


results_all = {}

score = {
    'Accuracy': 'accuracy',
    'F1': make_scorer(f1_score, average='binary'),
    'ROC-AUC': 'roc_auc'
}

In [ ]:
from core import (
    skf,
    X_train,
    y_train,
    evaluate_model,
)

from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd


cat = CatBoostClassifier(
    random_state=42,
    verbose=0,
    thread_count=-1,
)

param_dist_cat = {
    'iterations': [200, 500, 1000, 1500],
    'depth': [4, 6, 8, 10, 12],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7, 10, 15],
    'bagging_temperature': [0.0, 0.5, 1.0],
    'random_strength': [1, 3, 5, 10],
}

random_cat = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist_cat,
    n_iter=50,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_cat.fit(X_train, y_train)

best_cat = random_cat.best_estimator_
print(f'Лучшие параметры CatBoost: {random_cat.best_params_}')
print(f'Лучшая accuracy: {random_cat.best_score_:.4f}')

results_cat = evaluate_model(
    model=best_cat,
    X=X_train,
    y=y_train,
    cv=skf,
    scoring_dict=score
)

results_all['CatBoostClassifier'] = results_cat

df_results_2 = pd.DataFrame.from_dict(results_all, orient='index')
df_results_2 = df_results_2.reset_index().rename(columns={'index': 'Name'})

for col in ['Accuracy', 'F1', 'ROC-AUC']:
    if col in df_results_2.columns:
        df_results_2[col] = df_results_2[col].round(4)

print(df_results_2.to_markdown(index=False))